# Assignment 02 (Python), Sem 2520

## Introduction

This exercise utilises the same dataset as the R portion of the assignment: `FEV.DAT`. 

FEV is a measure of lung strength. The belief is that *exposure to smoking* would be associated with a *decrease* in FEV.

The following cell will read the data into your Python notebook. Do not edit it.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

fev = pd.read_table('data/FEV.DAT', sep='\\s+')

## Q1: Two Sample t-Test

Let $\mu_S$ be the average `FEV` for children who have been exposed to smoking, regardless of gender, age and height. Similarly, let $\mu_N$ be the average `FEV` for children who have **not** been exposed to smoking. Apply the two sample t-test to assess the following hypothesis, at 10% significance level:

\begin{eqnarray}
H_0 &:& \mu_S - \mu_N = 0\\
H_1 &:& \mu_S - \mu_N < 0
\end{eqnarray}

ADD CELLS AS NEEDED FOR Q1 HERE...

*If you feel that the assumptions for the test have not been met, you do not have to proceed to apply the corresponding non-parametric test.*



In [2]:
smokers = fev[fev["Smoke"] == 1]["FEV"]
non_smokers = fev[fev["Smoke"] == 0]["FEV"]

t_out = stats.ttest_ind(smokers, non_smokers, alternative="less")
t_out.pvalue
# p-value > 0.10, thus at 10% significance level, we fail to reject the null hypothesis.
# Thus, there is no statistically significant evidence that children exposed to smoking have
# lower FEV than those not exposed.

np.float64(0.9999999999003577)

## Q2: Robust Two Sample t-Test

The result of the above test seems counter to the intuition. An analyst suspects it is due to outliers in the data. As a result, they turn to a robust version of the test. This test, outlined in the references listed below, uses the trimmed mean instead of the usual mean. The test statistic is

$$
T_y = \frac{\bar{X}_{tS} - \bar{X}_{tN}}{\sqrt{d_S + d_N}}
$$

where 

* $\bar{X}_{tS}$ and $\bar{X}_{tN}$ correspond to the $\gamma=0.1$ trimmed means for the exposed (to smoking) and non-exposed groups respectively.
* $d_S$ and $d_N$ correspond to estimates of the square of standard error of the two groups. These are based on the Winsorised samples of the two groups.

The test statistic $T_y$ also follows a $t$ distribution, but with different degrees of freedom from earlier. 

* Compute the difference in trimmed means $\bar{X}_{tS} - \bar{X}_{tN}$ and save it as a numpy array named `diff_t_means` of shape `(1,)`.
* Read up on the documentation in `scipy.stats.ttest_ind` to understand how to call this version of the t-test. Extract the test statistic and the degrees of freedom and store them in a numpy array named `yuen_output` of shape `(2,)` with the first entry being the test statstic.

In [6]:
# YOUR CODE HERE
trim_gamma = 0.1

# difference in trimmed mean
trimmed_smokers = stats.trim_mean(smokers, proportiontocut= trim_gamma)
trimmed_non_smokers = stats.trim_mean(non_smokers, proportiontocut= trim_gamma)
diff_t_means = np.array([trimmed_smokers - trimmed_non_smokers])
diff_t_means

array([0.75104105])

In [7]:
# yuen_output
winsor_smokers = stats.mstats.winsorize(smokers, limits=trim_gamma)
winsor_non_smokers = stats.mstats.winsorize(non_smokers, limits=trim_gamma)

yuen_t_stats, p_value = stats.ttest_ind(winsor_smokers, winsor_non_smokers, alternative="less")
n_s = len(smokers)
n_n = len(non_smokers)
s_s = np.var(winsor_smokers, ddof=1)
s_n = np.var(winsor_non_smokers, ddof=1)
df = (s_s/n_s + s_n/n_n)**2 / ((s_s/n_s)**2/(n_s - 1) + (s_n/n_n)**2 / (n_n - 1))
yuen_output = np.array([yuen_t_stats, df])
yuen_output

array([ 8.15158987, 80.05865907])

## Note 

It looks like whether we use the robust or non-robust version of the test, we get the misleading result that exposure to smoking increases FEV. The issue is not related the test or outliers; it is an issue of not adjusting for other variables. Once we do so, like with the linear regression model in R, we see the model that reflects the relationship between smoking and lung performance.

## References

1. *The Two-Sample Trimmed t for Unequal Population Variances*, Karen K. Yuen, Biometrika, vo1. 1 (1974).
2. *Introduction to Robust Estimation and Hypothesis Testing* (pp. 157, chapter 5), Rand Wilcox, ISBN 978-0-12-386983-8, 3rd Edition (2013).